## Оценка качества извлечения

Идея бенчмарка состоит в измерении качества извлечения документов по описанию (docstring) типов, методов, функций и т.п., содержащихся в них, в зависимости от используемого алгортима сегментации исходных текстов. Сами docstring при этом удаляются из исходного кода. Помимо представленного алгоритма на основе AST, используются стандартные функции сегментации из LlamaIndex:

- `SentenceSplitter` - базовый алгоритм разбиения текстов на фиксированные сегменты.
- `CodeSplitter` - алгоритм, разбивающий код только по границам блоков кода.
- `BM25` - алгоритм Okapi BM25 (без сегментации, поиск осуществляется по всему документу).

In [ ]:
import time
import torch
import datasets
import pandas as pd
import numpy as np

from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, Settings, StorageContext, QueryBundle
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from tqdm.std import tqdm
from beir.retrieval.evaluation import EvaluateRetrieval
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine.retriever_query_engine import RetrieverQueryEngine
from llama_index.core.node_parser import CodeSplitter
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.retrievers.fusion_retriever import FUSION_MODES
from llama_index.core.postprocessor import SentenceTransformerRerank
from chunking import SourceCodeNodeParser
from openai import AsyncClient
from pydantic import BaseModel
import itertools
import asyncio


np.random.seed(42)

class Relevance(BaseModel):
    rate: int

Для оценки качества извлечения данных был собран [набор данных](https://huggingface.co/datasets/kmvi/code-ir-dataset), состоящий из трех блоков:

- `docs` - файлы исходного кода (документы) с удаленными из них docstring:
    - `doc_id` - идентификатор документа
    - `file` - имя файла
    - `content` - содержимое файла (с удаленными docstring's)
- `queries` - запросы к документам. В качестве запросов выступают docstring
    - `query_id` - идентификатор запроса
    - `query` - текст запроса
    - `type` - тип элемента кода, к которому относился docstring (тип, либо метод/функция)
- `qrels` - корректные пары запрос/документ
    - `query_id` - идентификатор запроса
    - `doc_id` - идентификатор документа

In [2]:
docs_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="docs")
queries_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="queries")
qrels_ds = datasets.load_dataset("kmvi/code-ir-dataset", name="qrels")

doc_ids = { row['file']: row['doc_id'] for row in docs_ds['train'] }
qrels = qrels_ds['train'].to_pandas().groupby('query_id').doc_id.apply(list).apply(lambda x: {k: 1 for k in x})

Подготавливаем исходные тексты для построения индекса.

In [3]:
docs = []
for row in docs_ds['train']:
    with open(row['file'], 'rt', encoding='utf-8') as f:
        text = f.read().replace('\r\n', '\n')

    doc = Document(id_=row['doc_id'], text=text, metadata={'file_name': row['file']})
    docs.append(doc)

Загрузка модели эмбеддингов:

In [ ]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    device="cuda",
    normalize=True,
    model_kwargs={'dtype': torch.bfloat16},
)

Settings.llm = None

LLM is explicitly disabled. Using MockLLM.


Построение индекса с использованием SentenceSplitter.

In [ ]:
client = QdrantClient(url='http://192.168.122.219:6333')
vector_store = QdrantVectorStore(client=client, collection_name='ir_base')
storage_context = StorageContext.from_defaults(vector_store=vector_store)
splitter = SentenceSplitter(chunk_size=256)
index = VectorStoreIndex.from_documents(docs, storage_context, transformations=[splitter], show_progress=True, store_nodes_override=True)
client.close()

Parsing nodes:   0%|          | 0/175 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/673 [00:00<?, ?it/s]

Построение индекса с использованием CodeSplitter:

In [ ]:
client = QdrantClient(url='http://192.168.122.219:6333')
vector_store = QdrantVectorStore(client=client, collection_name='ir_codesplitter')
storage_context = StorageContext.from_defaults(vector_store=vector_store)
splitter = CodeSplitter('csharp', max_chars=256)
index = VectorStoreIndex.from_documents(docs, storage_context, transformations=[splitter], show_progress=True, store_nodes_override=True)
client.close()

Parsing nodes:   0%|          | 0/175 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/276 [00:00<?, ?it/s]

Построение индекса с использованием нового алгоритма на основе AST:

In [ ]:
client = QdrantClient(url='http://192.168.122.219:6333')
vector_store = QdrantVectorStore(client=client, collection_name='ir_astsplitter')
storage_context = StorageContext.from_defaults(vector_store=vector_store)
parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)
index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True, store_nodes_override=True)
client.close()

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/969 [00:00<?, ?it/s]

## Вычисление метрик

Для расчета метрик используется функционал из библиотеки `beir`.

In [ ]:
retr = EvaluateRetrieval()
top_k = 10 # извлекаем 10 документов
client = QdrantClient(url='http://192.168.122.219:6333')

In [ ]:
storename = 'ir_base'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

Функция для расчета метрик. Также возвращает пары "запрос - извлеченные фрагменты", которые будут использоваться для оценки релевантности.

In [ ]:
def run_benchmark(engine):
    tm = time.perf_counter()
    results = dict()
    scores, retr_data = [], []
    for row in tqdm(queries_ds['train']):
        data = engine.retrieve(QueryBundle(row['query']))
        retr_data.append((row['query'], [node.text for node in data]))
        results[row['query_id']] = { doc_ids[node.metadata['file_name']]: node.score for node in data}
        scores.append(np.mean([node.score for node in data]))
    
    tm = time.perf_counter() - tm
    tm /= len(queries_ds['train'])

    metrics = retr.evaluate(qrels, results, k_values=[3,5,10])
    
    results = dict()
    for item in (*metrics, {'Latency': tm * 1000, 'Mean Score': np.mean(scores)}):
        results.update(item)

    return results, retr_data

In [ ]:
engine = index.as_query_engine(similarity_top_k=top_k)
metrics, retr_data = run_benchmark(engine)

100%|██████████| 1030/1030 [06:02<00:00,  2.85it/s]


In [ ]:
storename = 'ir_codesplitter'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

In [ ]:
engine = index.as_query_engine(similarity_top_k=top_k)
metrics_codesplit, retr_data_codesplit = run_benchmark(engine)

100%|██████████| 1030/1030 [08:03<00:00,  2.13it/s]


In [ ]:
storename = 'ir_astsplitter'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

In [ ]:
engine = index.as_query_engine(similarity_top_k=top_k)
metrics_ast, retr_data_ast = run_benchmark(engine)

100%|██████████| 1030/1030 [09:35<00:00,  1.79it/s]


In [ ]:
tm = time.perf_counter()
docids = list(doc_ids.values())
results_random = dict()
retr_data_random = []
for row in tqdm(queries_ds['train']):
    idx = np.random.choice(range(0, len(nodes)), top_k)
    results_random[row['query_id']] = { doc_ids[node.metadata['file_name']]: 1.0 / top_k for node in nodes[idx] }
    retr_data_random.append((row['query'], [node.text for node in nodes[idx]]))

tm = time.perf_counter() - tm
tm /= len(queries_ds['train'])

metrics_random = retr.evaluate(qrels, results_random, k_values=[3,5,10])
metrics_random = (*metrics_random, {'Latency': tm * 1000, 'Mean Score': -1})

results_random = dict()
for item in metrics_random:
    results_random.update(item)

metrics_random = results_random

100%|██████████| 1030/1030 [00:00<00:00, 5916.78it/s]


In [ ]:
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k)
engine = RetrieverQueryEngine.from_args(bm25, llm=Settings.llm)
metrics_bm25, retr_data_bm25 = run_benchmark(engine)

100%|██████████| 1030/1030 [00:01<00:00, 866.67it/s]


Дополнительно расчитаем релевантность извлеченных фрагментов исходному запросу с помощью LLM. Попросим LLM оценить релевантность по шкале от 0 до 10. Чтобы упростить получение ответа, используем структурный вывод. В качестве оценивающей LLM использовалась модель [Google Gemma 4](https://huggingface.co/google/gemma-4-26B-A4B-it).

In [ ]:
async def calc_relevance(retr_data):
    prompt_template = """You need to rate the relevance of the function or class description to its source code on a scale of 0 to 10.
    Description:
    {0}
    Source code:
    {1}
    """

    rates = []
    client = AsyncClient(base_url='http://localhost:8080', api_key='')
    batch_size = 4

    pg = tqdm(list(itertools.batched(retr_data, batch_size)))
    for batch in pg:
        tasks = []
        values = []
        for query, chunks in batch:
            for chunk in chunks[:1]:
                prompt = prompt_template.format(query, chunk)
                task = client.chat.completions.parse(
                    model='',
                    messages=[{'role': 'user', 'content': prompt}],
                    response_format=Relevance,
                    temperature=0, top_p=1, seed=42,
                )
                tasks.append(task)
            
        tasks = await asyncio.gather(*tasks)
        for result in tasks:
            msg = result.choices[0].message
            if not msg.refusal and msg.parsed:
                values.append(msg.parsed.rate)
        
        rates.append(np.mean(values))
        pg.set_postfix_str(np.mean(rates))

    await client.close()

    return np.mean(rates)

In [ ]:
rate_base = await calc_relevance(retr_data)

100%|██████████| 1030/1030 [32:22<00:00,  1.89s/it, 6.50873786407767]  


In [ ]:
rate_codesplit = await calc_relevance(retr_data_codesplit)

100%|██████████| 1030/1030 [25:36<00:00,  1.49s/it, 7.245631067961165] 


In [ ]:
rate_ast = await calc_relevance(retr_data_ast)

100%|██████████| 258/258 [12:12<00:00,  2.84s/it, 7.383720930232558] 
C:\Users\user\AppData\Local\Temp\ipykernel_24728\3661262904.py:1: RuntimeWarning: coroutine 'calc_relevance' was never awaited
  rate_ast = await calc_relevance(retr_data_ast)


In [ ]:
rate_random = await calc_relevance(retr_data_random)

100%|██████████| 258/258 [11:55<00:00,  2.77s/it, 0.49709302325581395]


In [ ]:
rate_bm25 = await calc_relevance(retr_data_bm25)

100%|██████████| 258/258 [12:37<00:00,  2.94s/it, 2.4612403100775193]


In [ ]:
metrics.update({'Relevance': rate_base / 10})
metrics_codesplit.update({'Relevance': rate_codesplit / 10})
metrics_ast.update({'Relevance': rate_ast / 10})
metrics_random.update({'Relevance': rate_random / 10})
metrics_bm25.update({'Relevance': rate_bm25 / 10})

Итоговый результат:

In [ ]:
rows = [metrics_random, metrics_bm25, metrics, metrics_codesplit, metrics_ast]

df = pd.DataFrame(rows, index=['Random', 'BM25', 'SentenceSplitter', 'CodeSplitter', 'AST Splitter'])
df = df[df.filter(regex='^NDCG|^Recall|^P|^La|^Me|^Re').columns]

def highlight_max(s, props=""):
    if s.name == "Latency":
        return np.where(s == np.min(s.values), props, "")
    else:
        return np.where(s == np.max(s.values), props, "")

def highlight_2nd_max(s, props=""):
    if s.name == "Latency":
        return np.where(s == np.unique(s.values)[1], props, "")
    else:
        return np.where(s == np.unique(s.values)[-2], props, "")

styled = df.style.apply(highlight_max, props="font-weight: bold;", axis=0) \
    .apply(highlight_2nd_max, props="text-decoration: underline;", axis=0) \
    .format(precision=4)

styled

,NDCG@3,NDCG@5,NDCG@10,Recall@3,Recall@5,Recall@10,P@3,P@5,P@10,Latency,Mean Score,Relevance
Random,0.0386,0.0601,0.0834,0.0511,0.1034,0.1714,0.0197,0.0227,0.0184,0.5991,-1.0000,0.0497
BM25,0.2138,0.2586,0.2768,0.2860,0.3938,0.4470,0.1074,0.0876,0.0494,0.6322,6.9776,0.2461
SentenceSplitter,0.5996,0.6172,0.6193,0.6884,0.7281,0.7339,0.2450,0.1573,0.0792,31.3442,0.5694,0.6509
CodeSplitter,0.5889,0.6265,0.6291,0.7203,0.8080,0.8153,0.2573,0.1742,0.0880,31.6780,0.6027,0.7246
AST Splitter,0.6104,0.6398,0.6455,0.7414,0.8106,0.8265,0.2657,0.1749,0.0892,32.2281,0.6195,0.7384


### Дополнительные подходы

Попробуем улучшить качество поиска с помощью следующих подходов:
- Совмещение результатов алгоритмов AST Splitter и BM25 (гибридный поиск)
- Применение реранжирования с помощью различных моделей: BAAI/bge-reranker-v2-m3, 
- Применение реранжирования и гибридного подхода

In [ ]:
storename = 'ir_astsplitter'
vector_store = QdrantVectorStore(client=client, collection_name=storename)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)

In [ ]:
parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)
retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever = QueryFusionRetriever(
    retrievers=[retriever, bm25],
    similarity_top_k=top_k,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm)

metrics_hybrid, retr_data_hybrid = run_benchmark(engine)
metrics_hybrid

100%|██████████| 1030/1030 [00:34<00:00, 30.13it/s]


{'NDCG@3': 0.34528,
 'NDCG@5': 0.44711,
 'NDCG@10': 0.47916,
 'MAP@3': 0.29489,
 'MAP@5': 0.35216,
 'MAP@10': 0.36762,
 'Recall@3': 0.47709,
 'Recall@5': 0.72421,
 'Recall@10': 0.8145,
 'P@3': 0.17379,
 'P@5': 0.1567,
 'P@10': 0.08816,
 'MRR@3': 0.32298,
 'MRR@5': 0.38011,
 'MRR@10': 0.39025,
 'Latency': 33.206727378775625,
 'Mean Score': 0.018186316329305285}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k)
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank_bge, retr_data_rerank_bge = run_benchmark(engine)
metrics_rerank_bge

100%|██████████| 1030/1030 [02:36<00:00,  6.59it/s]


{'NDCG@3': 0.62533,
 'NDCG@5': 0.65629,
 'NDCG@10': 0.66181,
 'MAP@3': 0.5814,
 'MAP@5': 0.60026,
 'MAP@10': 0.60297,
 'Recall@3': 0.73906,
 'Recall@5': 0.8111,
 'Recall@10': 0.82647,
 'P@3': 0.26472,
 'P@5': 0.17515,
 'P@10': 0.08922,
 'MRR@3': 0.59515,
 'MRR@5': 0.61087,
 'MRR@10': 0.61325,
 'Latency': 151.7253251456954,
 'Mean Score': 0.2360216}

In [ ]:
parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)

retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever2 = bm25
retriever = QueryFusionRetriever(
    retrievers=[retriever, retriever2],
    similarity_top_k=top_k+5,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank25_bge, retr_data_rerank25_bge = run_benchmark(engine)
metrics_rerank25_bge

DEBUG:bm25s:Building index from IDs objects
100%|██████████| 1030/1030 [04:44<00:00,  3.62it/s]


{'NDCG@3': 0.55699,
 'NDCG@5': 0.60369,
 'NDCG@10': 0.61529,
 'MAP@3': 0.5063,
 'MAP@5': 0.53459,
 'MAP@10': 0.54009,
 'Recall@3': 0.68634,
 'Recall@5': 0.79608,
 'Recall@10': 0.82909,
 'P@3': 0.24693,
 'P@5': 0.17262,
 'P@10': 0.0899,
 'MRR@3': 0.52298,
 'MRR@5': 0.54715,
 'MRR@10': 0.55194,
 'Latency': 276.3970538835918,
 'Mean Score': 0.2478097}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k)
reranker = SentenceTransformerRerank(model="cross-encoder/stsb-roberta-large", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank_roberta, retr_data_rerank_roberta = run_benchmark(engine)
metrics_rerank_roberta

100%|██████████| 1030/1030 [03:04<00:00,  5.58it/s]


{'NDCG@3': 0.59918,
 'NDCG@5': 0.63165,
 'NDCG@10': 0.63349,
 'MAP@3': 0.54543,
 'MAP@5': 0.56464,
 'MAP@10': 0.56557,
 'Recall@3': 0.7444,
 'Recall@5': 0.82129,
 'Recall@10': 0.82647,
 'P@3': 0.26764,
 'P@5': 0.17728,
 'P@10': 0.08922,
 'MRR@3': 0.55421,
 'MRR@5': 0.57125,
 'MRR@10': 0.57201,
 'Latency': 179.07639922330029,
 'Mean Score': 0.47242284}

In [ ]:
parser = SourceCodeNodeParser(chunk_size=256)
nodes = parser(docs)

retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever2 = bm25
retriever = QueryFusionRetriever(
    retrievers=[retriever, retriever2],
    similarity_top_k=top_k+5,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)
reranker = SentenceTransformerRerank(model="cross-encoder/stsb-roberta-large", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank25_bert, retr_data_rerank25_bert = run_benchmark(engine)
metrics_rerank25_bert

DEBUG:bm25s:Building index from IDs objects
100%|██████████| 1030/1030 [05:58<00:00,  2.87it/s]


{'NDCG@3': 0.33075,
 'NDCG@5': 0.44803,
 'NDCG@10': 0.47345,
 'MAP@3': 0.27725,
 'MAP@5': 0.34295,
 'MAP@10': 0.35528,
 'Recall@3': 0.47712,
 'Recall@5': 0.75935,
 'Recall@10': 0.83023,
 'P@3': 0.1712,
 'P@5': 0.1635,
 'P@10': 0.08981,
 'MRR@3': 0.28511,
 'MRR@5': 0.34977,
 'MRR@10': 0.36043,
 'Latency': 348.33180155329296,
 'Mean Score': 0.50293595}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k)
reranker = SentenceTransformerRerank(model="Qwen/Qwen3-Reranker-0.6B", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank_qwen, retr_data_qwen = run_benchmark(engine)
metrics_rerank_qwen

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-Reranker-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 1030/1030 [03:19<00:00,  5.17it/s]


{'NDCG@3': 0.53853,
 'NDCG@5': 0.58738,
 'NDCG@10': 0.59495,
 'MAP@3': 0.48177,
 'MAP@5': 0.5107,
 'MAP@10': 0.51445,
 'Recall@3': 0.68987,
 'Recall@5': 0.80528,
 'Recall@10': 0.82647,
 'P@3': 0.24757,
 'P@5': 0.17359,
 'P@10': 0.08922,
 'MRR@3': 0.49401,
 'MRR@5': 0.51979,
 'MRR@10': 0.52282,
 'Latency': 193.28136621359553,
 'Mean Score': 0.677955}

In [ ]:
retriever = index.as_retriever(similarity_top_k=top_k+5)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k+5)
retriever2 = bm25
retriever = QueryFusionRetriever(
    retrievers=[retriever, retriever2],
    similarity_top_k=top_k+5,
    use_async=False, verbose=False, mode=FUSION_MODES.RECIPROCAL_RANK, num_queries=1)
reranker = SentenceTransformerRerank(model="Qwen/Qwen3-Reranker-0.6B", top_n=top_k)
reranker._model.config.pad_token_id = reranker._model.config.eos_token_id
reranker._model.model.config.pad_token_id = reranker._model.model.config.eos_token_id
postprocessors = [reranker]

engine = RetrieverQueryEngine.from_args(retriever, llm=Settings.llm, node_postprocessors=postprocessors)

metrics_rerank25_qwen, retr_data_rerank25_qwen = run_benchmark(engine)
metrics_rerank25_qwen

DEBUG:bm25s:Building index from IDs objects
Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-Reranker-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 1030/1030 [06:21<00:00,  2.70it/s]


{'NDCG@3': 0.23268,
 'NDCG@5': 0.36063,
 'NDCG@10': 0.40707,
 'MAP@3': 0.19246,
 'MAP@5': 0.26442,
 'MAP@10': 0.28592,
 'Recall@3': 0.33537,
 'Recall@5': 0.64479,
 'Recall@10': 0.77715,
 'P@3': 0.1246,
 'P@5': 0.13942,
 'P@10': 0.08398,
 'MRR@3': 0.20599,
 'MRR@5': 0.27565,
 'MRR@10': 0.29585,
 'Latency': 370.57326844649265,
 'Mean Score': 0.2542915}

Оценка релевантности поиска с помощью LLM:

In [ ]:
rate_hybrid = await calc_relevance(retr_data_hybrid)

100%|██████████| 258/258 [11:59<00:00,  2.79s/it, 6.853682170542636] 
C:\Users\user\AppData\Local\Temp\ipykernel_18208\874952637.py:1: RuntimeWarning: coroutine 'calc_relevance' was never awaited
  rate_hybrid = await calc_relevance(retr_data_hybrid)


In [ ]:
rate_rerank25 = await calc_relevance(retr_data_rerank25_bge)

100%|██████████| 258/258 [12:14<00:00,  2.85s/it, 6.996124031007752] 
C:\Users\user\AppData\Local\Temp\ipykernel_18208\2031811117.py:1: RuntimeWarning: coroutine 'calc_relevance' was never awaited
  rate_rerank25 = await calc_relevance(retr_data_rerank25_bge)


In [ ]:
rate_rerank = await calc_relevance(retr_data_rerank_bge)

100%|██████████| 258/258 [11:55<00:00,  2.77s/it, 7.508720930232558] 


In [ ]:
rate_qwen = await calc_relevance(retr_data_qwen)

100%|██████████| 258/258 [11:35<00:00,  2.69s/it, 5.238372093023256] 


In [ ]:
rate_roberta = await calc_relevance(retr_data_rerank_roberta)

100%|██████████| 258/258 [11:57<00:00,  2.78s/it, 6.410852713178294] 


In [ ]:
rate_roberta25 = await calc_relevance(retr_data_rerank25_bert)

100%|██████████| 258/258 [12:58<00:00,  3.02s/it, 4.234496124031008] 


In [ ]:
rate_qwen25 = await calc_relevance(retr_data_rerank25_qwen)

100%|██████████| 258/258 [12:54<00:00,  3.00s/it, 3.618217054263566] 


In [ ]:
metrics_hybrid.update({'Relevance': rate_hybrid / 10})
metrics_rerank_bge.update({'Relevance': rate_rerank / 10})
metrics_rerank25_bge.update({'Relevance': rate_rerank25 / 10})
metrics_rerank_qwen.update({'Relevance': rate_qwen / 10})
metrics_rerank_roberta.update({'Relevance': rate_roberta / 10})
metrics_rerank25_bert.update({'Relevance': rate_roberta25 / 10})
metrics_rerank25_qwen.update({'Relevance': rate_qwen25 / 10})

Итоговый результат:

In [ ]:
rows = [metrics_hybrid, metrics_rerank25_bge, metrics_rerank_roberta, metrics_rerank_qwen, metrics_rerank_bge, metrics_rerank25_bert, metrics_rerank25_qwen]
df = pd.DataFrame(rows, index=['AST Splitter + BM25', 'AST Splitter + BM25 + BGE',
                               'AST Splitter + RoBERTa', 'AST Splitter + Qwen', 'AST Splitter + BGE',
                               'AST Splitter + BM25 + RoBERTa', 'AST Splitter + BM25 + Qwen'])
df = df[df.filter(regex='^NDCG|^Recall|^P|^La|^Me|^Re').columns]

def highlight_max(s, props=""):
    if s.name == "Latency":
        return np.where(s == np.min(s.values), props, "")
    else:
        return np.where(s == np.max(s.values), props, "")

def highlight_2nd_max(s, props=""):
    if s.name == "Latency":
        return np.where(s == np.unique(s.values)[1], props, "")
    else:
        return np.where(s == np.unique(s.values)[-2], props, "")

styled = df.style.apply(highlight_max, props="font-weight: bold;", axis=0) \
    .apply(highlight_2nd_max, props="text-decoration: underline;", axis=0) \
    .format(precision=4)

styled

,NDCG@3,NDCG@5,NDCG@10,Recall@3,Recall@5,Recall@10,P@3,P@5,P@10,Latency,Mean Score,Relevance
AST Splitter + BM25,0.3453,0.4471,0.4792,0.4771,0.7242,0.8145,0.1738,0.1567,0.0882,33.2067,0.0182,0.6854
AST Splitter + BM25 + BGE,0.5570,0.6037,0.6153,0.6863,0.7961,0.8291,0.2469,0.1726,0.0899,276.3971,0.2478,0.6996
AST Splitter + RoBERTa,0.5992,0.6317,0.6335,0.7444,0.8213,0.8265,0.2676,0.1773,0.0892,179.0764,0.4724,0.6411
AST Splitter + Qwen,0.5385,0.5874,0.5949,0.6899,0.8053,0.8265,0.2476,0.1736,0.0892,193.2814,0.6780,0.5238
AST Splitter + BGE,0.6253,0.6563,0.6618,0.7391,0.8111,0.8265,0.2647,0.1752,0.0892,151.7253,0.2360,0.7509
AST Splitter + BM25 + RoBERTa,0.3307,0.4480,0.4734,0.4771,0.7593,0.8302,0.1712,0.1635,0.0898,348.3318,0.5029,0.4234
AST Splitter + BM25 + Qwen,0.2327,0.3606,0.4071,0.3354,0.6448,0.7772,0.1246,0.1394,0.0840,370.5733,0.2543,0.3618
